In [1]:
# !pip install pandas

In [1]:
import json
def load_json(filename):
    """
    Load a JSON file given a filename
    If the file doesn't exist, then return an empty dictionary instead
    """
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {}

In [2]:
import pandas as pd

In [3]:
import glob
import os

# Find all files in the raw questions directory
raw_files = glob.glob('../../tasks/ttct/data/raw/*.csv')

question_to_group = {}

for file_path in raw_files:
    group_name = os.path.basename(file_path)
    tmp_df = pd.read_csv(file_path)
    for q in tmp_df['CoT_prompt'].tolist():
        question_to_group[q] = group_name
# question_to_group is now a dictionary mapping each question to its group (filename)

In [4]:
# tmp_df

In [5]:
len(question_to_group)

697

In [6]:
# group_keys

In [7]:
import pandas as pd
import glob
import random

# Load all evaluation files for gpt-4.1
files = glob.glob('../../tasks/ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv')
df_list = [pd.read_csv(f) for f in files]
df = pd.concat(df_list, ignore_index=True)

# Keep only the required columns
df = df[['infer_cot_input', 'infer_cot_pred', 'eval_cot_input', 'eval_cot_pred']]

# Group rows by the prefix of 'infer_cot_input' (first sentence or instruction)
def get_group_key(text):
    # return text.split('.')[0].strip()  # adjust if instructions are not sentence-based
    # return text.strip()[:20]
    if text in question_to_group:
        return question_to_group[text]
    else:     
        return "unknown"

df['group'] = df['infer_cot_input'].apply(get_group_key)
group_keys = df['group'].unique()

# Sample 5 rows per group
sampled_rows = []
for key in group_keys:
    if key == "unknown":
        continue
    group_df = df[df['group'] == key]
    sampled = group_df.sample(n=min(5, len(group_df)), random_state=42)
    sampled_rows.append(sampled)

sampled_df = pd.concat(sampled_rows).drop(columns=['group']).reset_index(drop=True)

# Save or display the sampled dataframe
# print(sampled_df)
sampled_df.shape

(35, 4)

In [8]:
# Load the dataframe from the specified CSV file
df_gpt4 = pd.read_csv('../../tasks/ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv')

# Generate mapping from row index to group name using get_group_key
row_to_group = {idx: get_group_key(row['infer_cot_input']) for idx, row in df_gpt4.iterrows()}
len(row_to_group)

700

In [9]:
# row_to_group

In [10]:
# eval_dir = '../../ttct/data/evaluations/temp_1'
# csv_files = glob.glob(os.path.join(eval_dir, '*.csv'))

# for file in csv_files:
#     df_tmp = pd.read_csv(file)
#     print(f"{os.path.basename(file)}: {df_tmp.shape}")

In [11]:
input_questions = pd.read_csv('../../tasks/ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv')['infer_cot_input'].tolist()

def sample_evaluation_file_sequential(filepath, n_per_group=5, group_size=100, random_state=42):
    print(f"Sampling from {filepath}")
    df = pd.read_csv(filepath)
    if 'infer_cot_input' not in df.columns:
        df['infer_cot_input'] = input_questions[:len(df)]
    df['group'] = [row_to_group[i] for i in df.index]
    sampled_rows = []
    for group_name in df['group'].unique():
        group_df = df[df['group'] == group_name]
        sampled = group_df.sample(n=min(n_per_group, len(group_df)), random_state=random_state)
        sampled_rows.append(sampled)
    sampled_df = pd.concat(sampled_rows).reset_index(drop=True)
    return sampled_df

In [12]:
# sample_evaluation_file_sequential('../../ttct/data/evaluations/temp_1/Qwen2.5-72B-Instruct.csv',).shape

In [13]:
# List of evaluation files to sample from
eval_files = [
    '../../tasks/ttct/data/evaluations/temp_1/Qwen2.5-7B-Instruct.csv',
    '../../tasks/ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv',
    '../../tasks/ttct/data/evaluations/temp_1/OLMo-2-1124-13B-Instruct.csv'
]


# Sample 35 rows per file and concatenate results
required_columns = ['infer_cot_input', 'infer_cot_pred', 'eval_cot_input', 'eval_cot_pred']
sampled_all = []
for filepath in eval_files:
    sampled = sample_evaluation_file_sequential(filepath, n_per_group=3)
    # print('Before filtering, columns for', file_path, sampled.columns)
    sampled = sampled[required_columns]
    print('Columns for', file_path, sampled.columns, len(sampled.columns))
    sampled['source_file'] = os.path.basename(filepath)
    print(f"Sampled {sampled.shape[0]} rows from {os.path.basename(filepath)}")
    sampled_all.append(sampled)

big_table = pd.concat(sampled_all, ignore_index=True)
big_table.shape  # Should be (105, columns)

Sampling from ../../tasks/ttct/data/evaluations/temp_1/Qwen2.5-7B-Instruct.csv
Columns for ../../tasks/ttct/data/raw/im_task_prompt.csv Index(['infer_cot_input', 'infer_cot_pred', 'eval_cot_input', 'eval_cot_pred'], dtype='object') 4
Sampled 21 rows from Qwen2.5-7B-Instruct.csv
Sampling from ../../tasks/ttct/data/evaluations/temp_1/gpt-4.1-2025-04-14.csv
Columns for ../../tasks/ttct/data/raw/im_task_prompt.csv Index(['infer_cot_input', 'infer_cot_pred', 'eval_cot_input', 'eval_cot_pred'], dtype='object') 4
Sampled 21 rows from gpt-4.1-2025-04-14.csv
Sampling from ../../tasks/ttct/data/evaluations/temp_1/OLMo-2-1124-13B-Instruct.csv
Columns for ../../tasks/ttct/data/raw/im_task_prompt.csv Index(['infer_cot_input', 'infer_cot_pred', 'eval_cot_input', 'eval_cot_pred'], dtype='object') 4
Sampled 21 rows from OLMo-2-1124-13B-Instruct.csv


(63, 5)

In [14]:
def question_type(input_text):
    if "Common Problems Task" in input_text:
        return "5_common_problems"
    elif "What would happen if" in input_text:
        return "2_consequences"
    elif "Creativity Improvement Task" in input_text:
        return "6_improvement"
    elif "to construct a narrative or story" in input_text:
        return "7_story"
    elif "Unusual Uses Task" in input_text:
        return "1_unusual_uses"
    elif input_text.startswith("If "):
        return "4_situation"
    elif "Just suppose you " in input_text:
        return "3_just_suppose"
    else:
        return "unknown"

In [15]:
big_table['question_type'] = big_table['infer_cot_input'].apply(question_type)
big_table.question_type.value_counts()

question_type
5_common_problems    9
2_consequences       9
6_improvement        9
7_story              9
3_just_suppose       9
4_situation          9
1_unusual_uses       9
Name: count, dtype: int64

In [16]:
ttct_rubric_texts = {}
for i in range(1, 8):
    # read rubric file from ../data/ttct_rubrics/{i}.txt 
    with open(f'../data/ttct_rubrics/{i}.txt', 'r') as f:
        rubric = f.read()
    ttct_rubric_texts[i] = rubric

In [25]:
big_table['rubric'] = big_table['question_type'].apply(lambda x: ttct_rubric_texts[int(x.split('_')[0])])
big_table['tmp_id'] = range(len(big_table))
big_table.head()

,infer_cot_input,infer_cot_pred,eval_cot_input,eval_cot_pred,source_file,question_type,rubric,ttct_id,tmp_id
0,"Common Problems Task. In this task, you will b...",Let's break this down step by step to identif...,You are an expert of psychology. Your objectiv...,Fluency: 4\nFlexibility: 4\nOriginality: 4\nE...,Qwen2.5-7B-Instruct.csv,5_common_problems,- Definition: This task focuses on everyday pr...,0,0
1,"Common Problems Task. In this task, you will b...","Sure, let's break down the scenario of writin...",You are an expert of psychology. Your objectiv...,FLUENCY: 4 \nFLEXIBILITY: 4 \nORIGINALITY: ...,Qwen2.5-7B-Instruct.csv,5_common_problems,- Definition: This task focuses on everyday pr...,1,1
2,"Common Problems Task. In this task, you will b...",This is a complex issue involving many potent...,You are an expert of psychology. Your objectiv...,Fluency: 4 \n Flexibility: 4 \n Originality: ...,Qwen2.5-7B-Instruct.csv,5_common_problems,- Definition: This task focuses on everyday pr...,2,2
3,What would happen if the world's deserts sudde...,If the world's deserts suddenly turned into f...,You are an expert of psychology. Your objectiv...,Flexibility: 4\n Originality: 4\n Elaboration...,Qwen2.5-7B-Instruct.csv,2_consequences,- Definition: This task focuses on the ability...,3,3
4,What would happen if plastic could biodegrade ...,"Sure, let's think through this step by step:\...",You are an expert of psychology. Your objectiv...,Fluency: 3\nFlexibility: 4\nOriginality: 3\nE...,Qwen2.5-7B-Instruct.csv,2_consequences,- Definition: This task focuses on the ability...,4,4


- Save sample for MTurk pilot

In [22]:
import copy
pilot_data = []
for q in big_table['question_type'].unique():
    tmp_data = copy.deepcopy(big_table.loc[big_table['question_type'] == q]).head(1)
    pilot_data.append(tmp_data)
pilot_data_df = pd.concat(pilot_data).reset_index(drop=True).drop(columns=['source_file', 'eval_cot_input', 'eval_cot_pred'])
pilot_data_df.shape

(7, 4)

In [64]:
pilot_data_df.to_csv('../data/mturk_input/ttct_pilot_data.csv', index=False)

In [23]:
pilot_data_df

,infer_cot_input,infer_cot_pred,question_type,rubric
0,"Common Problems Task. In this task, you will b...",Let's break this down step by step to identif...,5_common_problems,- Definition: This task focuses on everyday pr...
1,What would happen if the world's deserts sudde...,If the world's deserts suddenly turned into f...,2_consequences,- Definition: This task focuses on the ability...
2,Creativity Improvement Task. You'll be present...,. The shopping cart is a simple utility that h...,6_improvement,- Definition: Improvement. This task focuses o...
3,You are to construct a narrative or story base...,.\n\nI understand. Let's construct a story bas...,7_story,- Definition: maginative stories. This task is...
4,Just suppose you could decide the length of a ...,"To decide the length of a day, we need to con...",3_just_suppose,- Definition: This task encourages imaginative...
5,"If there were no more night-time, how would yo...","Without night, we would still sleep, but our ...",4_situation,- Definition: This task is designed to assess ...
6,Unusual Uses Task. You will be presented with ...,.\nCertainly! Let's think through this step by...,1_unusual_uses,- Definition: This task challenges individuals...


- Save in batch

In [26]:
big_table.columns

Index(['infer_cot_input', 'infer_cot_pred', 'eval_cot_input', 'eval_cot_pred',
       'source_file', 'question_type', 'rubric', 'ttct_id', 'tmp_id'],
      dtype='object')

In [31]:
start = 0
end = big_table.shape[0]
step =7

while start < end:
    batch_df = big_table.iloc[start:start+step][['infer_cot_input', 'infer_cot_pred', 'rubric']]
    batch_number = (start // step) + 1
    batch_df.to_csv(f'../data/mturk_input/ttct/ttct_sampled_{batch_number}.csv', index=False)
    print(f"Saved batch {batch_number} with {batch_df.shape} rows.")
    start += step
# big_table.head(15).drop(columns=['eval_cot_input', 'eval_cot_pred']).to_csv('../data/mturk_input/ttct_sampled_1.csv', index=False)

big_table.to_csv('../data/mturk_input/ttct/ttct_full_data.csv', index=False)

Saved batch 1 with (7, 3) rows.
Saved batch 2 with (7, 3) rows.
Saved batch 3 with (7, 3) rows.
Saved batch 4 with (7, 3) rows.
Saved batch 5 with (7, 3) rows.
Saved batch 6 with (7, 3) rows.
Saved batch 7 with (7, 3) rows.
Saved batch 8 with (7, 3) rows.
Saved batch 9 with (7, 3) rows.


In [52]:
# big_table.to_csv(f'../data/mturk_input/ttct_sampled_all.csv', index=False)

In [55]:
batch_df

,infer_cot_input,infer_cot_pred,eval_cot_input,eval_cot_pred,source_file,question_type,rubric
60,Unusual Uses Task. You will be presented with ...,.\n\nSOLUTION: clean carpets (in a backwards ...,You are an expert of psychology. Your objectiv...,Fluency: 10\nFlexibility: 8\nOriginality: 7\n...,OLMo-2-1124-13B-Instruct.csv,1_unusual_uses,- Definition: This task challenges individuals...
61,Unusual Uses Task. You will be presented with ...,", one per line.\na frisbee\nthrow at people yo...",You are an expert of psychology. Your objectiv...,Fluency: 5\nOriginality: 5\nFlexibility: 5\nE...,OLMo-2-1124-13B-Instruct.csv,1_unusual_uses,- Definition: This task challenges individuals...
62,Unusual Uses Task. You will be presented with ...,.\n\nSolution: Put hats/gloves/boot on it. Ha...,You are an expert of psychology. Your objectiv...,Fluency: 5\nFlexibility: 5\nOriginality: 4\nE...,OLMo-2-1124-13B-Instruct.csv,1_unusual_uses,- Definition: This task challenges individuals...


In [45]:
qsf_ttct_dummy = load_json('../data/mturk_input/TTCT.qsf')

In [47]:
qsf_ttct_dummy

{'SurveyEntry': {'SurveyID': 'SV_ebAgY4hwuVLkiPQ',
  'SurveyName': 'TTCT',
  'SurveyDescription': None,
  'SurveyOwnerID': 'UR_0Jl2b39GcMgDGVE',
  'SurveyBrandID': 'pitt',
  'DivisionID': None,
  'SurveyLanguage': 'EN',
  'SurveyActiveResponseSet': 'RS_a47x2FqnHj4fjeK',
  'SurveyStatus': 'Active',
  'SurveyStartDate': '0000-00-00 00:00:00',
  'SurveyExpirationDate': '0000-00-00 00:00:00',
  'SurveyCreationDate': '2025-10-09 09:54:54',
  'CreatorID': 'UR_0Jl2b39GcMgDGVE',
  'LastModified': '2025-10-09 10:07:07',
  'LastAccessed': '0000-00-00 00:00:00',
  'LastActivated': '2025-10-09 10:06:56',
  'Deleted': None},
 'SurveyElements': [{'SurveyID': 'SV_ebAgY4hwuVLkiPQ',
   'Element': 'BL',
   'PrimaryAttribute': 'Survey Blocks',
   'SecondaryAttribute': None,
   'TertiaryAttribute': None,
   'Payload': [{'Type': 'Default',
     'Description': 'Question block',
     'ID': 'BL_1YSHN0Ba9tGuiou',
     'BlockElements': [{'Type': 'Question', 'QuestionID': 'QID4'},
      {'Type': 'Question', 'Que

In [27]:
print(sampled_df['infer_cot_input'].values[0])

Common Problems Task. In this task, you will be presented with a scenario or situation. Your job is to think about it and identify as many potential problems or issues that may arise in connection with each situation. Let's think step by step. The scenario is: Pioneering a mission to the bottom of the ocean.


In [28]:
print(sampled_df['infer_cot_pred'].values[0])

Absolutely! Let’s break down the scenario of pioneering a mission to the bottom of the ocean and identify as many potential problems or issues as possible, step by step:

**1. Technical Challenges**  
- **Extreme water pressure:** The pressure at the bottom of the ocean (10,000+ meters) is crushing, requiring specially engineered vehicles and materials.
- **Engineering submersibles:** Building submersibles tough enough to survive the pressure while remaining maneuverable and lightweight.
- **Communication limitations:** Radio waves don't travel well through water, so maintaining contact with the surface or mission control is difficult.
- **Power supply:** Supplying and storing sufficient power for propulsion, life support, lights, and data-gathering equipment over potentially extended missions.
- **Navigation:** GPS does not work underwater; alternative navigation methods (acoustic, inertial) are required but less accurate.
- **Equipment failure:** The harsh conditions may cause mechan

In [29]:
print(sampled_df['eval_cot_input'].values[0])

You are an expert of psychology. Your objective is to assess the subject’s creativity through their answers to some question/answering task related to divergent thinking.
You will be given a question-answer pair. Your task is to score the answer.
You should rate the answer on five metrics. For all five metrics, assign a score between 1 and 5, with 5 being the highest. Five metrics are:
1. Fluency. Fluency refers to the ability to generate a large quantity of ideas or solutions to a given problem. This measure isn’t concerned with the quality or uniqueness of the ideas, but rather the sheer volume. The more ideas one can produce, the higher the fluency.
2. Flexibility. Flexibility is the capacity to shift one’s thinking and to produce a wide range of ideas from different categories or perspectives. It involves being able to think outside of the box and to switch from one type of idea to another.
3. Originality. Originality refers to the ability to come up with unique or novel ideas that

In [30]:
print(sampled_df['eval_cot_pred'].values[0])

 Fluency: 5 Flexibility: 5 Originality: 4 Elaboration: 5 Overall: 5
Fluency: 5  
Flexibility: 5  
Originality: 4  
Elaboration: 5  
Overall: 5
